# 05 - First Model: Logistic Regression

**Goal for today:** load our processed data, split it properly, train our first
classifier, and evaluate it with metrics that actually make sense given the class
imbalance we found back in notebook 01 (~73% No / ~27% Yes).

**Why this matters for MLA-C01 (Domain 2 - ML Model Development):**
This is the core of Domain 2 - selecting an approach, training it correctly (with a
proper train/test split to avoid fooling yourself), and evaluating it with the right
metrics. Accuracy alone is dangerously misleading on imbalanced data, and knowing why
- and what to use instead - is one of the most exam-relevant things you'll learn in
this whole project.

## Step 0: Load the processed data from notebook 04

**What this cell does:** loads the file we saved at the end of notebook 04 - already
fully numeric, zero missing values, ready to go. This is exactly why we saved it
separately instead of re-deriving it here: notebook 05's job is modeling, not
re-doing data cleaning.

In [1]:
import pandas as pd

df = pd.read_csv('../data/telco_churn_processed.csv')
df.shape

(7043, 31)

## Step 1: Separate features (X) from target (y)

**What this cell does:** by ML convention, `X` (capital, since it's typically a 2D
table) holds every column the model is allowed to look at, and `y` (lowercase, a
single column) holds the answer we want it to predict. `.drop(columns=['Churn'])`
removes the target from the feature set - a model must never see the answer it's
trying to predict, or it isn't really learning anything.

In [2]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X.shape, y.shape

((7043, 30), (7043,))

## Step 2: Split into training and test sets

**What this cell does:** `train_test_split` randomly divides your data into two
separate chunks. The model will only ever see the **training set** during learning.
The **test set** is held back completely and only used afterward, to check how well
the model performs on data it has genuinely never seen.

**Why this matters so much:** if you evaluated a model on the same data it trained
on, you'd be measuring memorization, not generalization - like grading a student on
the exact questions they already saw the answers to. A model can look excellent on
training data and still fail badly on new, real-world customers. The train/test split
is what protects you from that illusion, and it's one of the single most
exam-relevant concepts in Domain 2.

**Parameters used:**
- `test_size=0.2` - hold back 20% of rows for testing, train on the remaining 80%
- `random_state=42` - a fixed "seed" so the random split is reproducible. Anyone
  re-running this notebook with the same seed gets the exact same split, which matters
  for comparing results fairly later.
- `stratify=y` - ensures both the training and test sets preserve the same ~73/27
  churn ratio as the full dataset. Without this, a random split could accidentally
  put very few churners in the test set purely by chance, given the imbalance.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((5634, 30), (1409, 30))

In [5]:
print('Churn rate in training set:', y_train.mean())
print('Churn rate in test set:', y_test.mean())

Churn rate in training set: 0.2653532126375577
Churn rate in test set: 0.2654364797728886


Both should be almost identical (~26.5%) and close to the full dataset's ~26.5% -
that's `stratify=y` doing its job.

## Step 3: Scale the features

**What this cell does:** `StandardScaler` transforms every numeric column so it has
a mean of 0 and a standard deviation of 1 - putting all features on a comparable
scale.

**Why this matters for Logistic Regression specifically:** look at the difference in
scale between `tenure` (0-72) and `TotalCharges` (0-8684) in our raw features. Some
algorithms - Logistic Regression among them - are sensitive to this, because they work
by finding weighted combinations of features, and a feature with naturally larger
numbers can dominate the math even if it isn't actually more important. Scaling
levels the playing field so the model judges features by their actual relationship to
churn, not by which one happens to have bigger raw numbers.

**Important detail:** we call `.fit_transform()` on the *training* set only, but
`.transform()` (no fit) on the *test* set. Fitting the scaler learns the mean/std from
training data only - if we let it see the test set too, that would leak information
the model shouldn't have access to at training time. This is the same "don't let the
model see the answer key" principle from Step 2, applied to preprocessing.

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Step 4: Train a Logistic Regression model

**What this cell does:** `LogisticRegression` is a good first-choice algorithm for
binary classification - simple, fast, and its output is directly interpretable as a
probability. `.fit(X_train_scaled, y_train)` is where the actual learning happens -
the model looks at the training features and training answers together, and adjusts
its internal weights to best separate churners from non-churners.

We're deliberately starting simple. Domain 2 expects you to know several algorithm
families, but you always want a simple, fast baseline first - it tells you whether a
more complex model is actually worth the extra cost, and gives you something concrete
to compare against.

In [9]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lb

## Step 5: Generate predictions on the test set

**What this cell does:** two different kinds of output from the same trained model:
- `.predict()` returns a hard 0/1 class prediction for each test row
- `.predict_proba()` returns the underlying probability the model assigned - we grab
  column `[:, 1]`, which is the predicted probability of class `1` (churn)

Keeping both matters: the hard 0/1 prediction is what you'd act on directly, but the
probability score is more informative and is what a metric like ROC-AUC (next step)
is actually built on.

In [10]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

y_pred[:10]

array([0, 1, 0, 0, 0, 1, 0, 0, 0, 0])

## Step 6: Evaluate - and why accuracy alone is misleading here

**Before running anything, think about this:** if a lazy model just predicted "No
churn" for every single customer, it would be right about 73% of the time - because
that's the actual No/Yes split in this data. **A model can score 73% "accuracy"
while having learned nothing at all.** That's exactly the trap accuracy sets on
imbalanced data, and exactly why Domain 2 pushes you toward a fuller set of metrics.

In [11]:
from sklearn.metrics import accuracy_score

baseline_accuracy = 1 - y_test.mean()
model_accuracy = accuracy_score(y_test, y_pred)

print(f'"Always predict No" baseline accuracy: {baseline_accuracy:.4f}')
print(f'Our model accuracy:                    {model_accuracy:.4f}')

"Always predict No" baseline accuracy: 0.7346
Our model accuracy:                    0.8070


Our model should beat that lazy baseline, but the margin is smaller than accuracy
alone makes it look. To really understand what's happening, we need to see *where*
the model is right and wrong - which brings us to the confusion matrix.

## Step 7: Confusion matrix

**What this cell does:** a confusion matrix is a 2x2 table breaking every prediction
into one of four outcomes:
- **True Negative (top-left)** - actually didn't churn, correctly predicted No
- **False Positive (top-right)** - actually didn't churn, incorrectly predicted Yes
- **False Negative (bottom-left)** - actually churned, incorrectly predicted No
- **True Positive (bottom-right)** - actually churned, correctly predicted Yes

For a churn model, **False Negatives are usually the costly mistake** - that's a
customer about to leave who the model told you not to worry about, so nobody
intervenes and the company loses them. Understanding which type of error matters most
for your specific business problem is a real Domain 2 skill, not just a math exercise.

In [12]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
pd.DataFrame(
    cm,
    index=['Actual: No Churn', 'Actual: Churn'],
    columns=['Predicted: No Churn', 'Predicted: Churn']
)

,Predicted: No Churn,Predicted: Churn
Actual: No Churn,925,110
Actual: Churn,162,212


## Step 8: Precision, Recall, F1, and ROC-AUC

**What these metrics mean, and why we need more than one:**

- **Precision** - of everyone the model predicted would churn, what fraction actually
  did? High precision means few false alarms.
- **Recall** - of everyone who actually churned, what fraction did the model catch?
  High recall means few missed churners (few False Negatives).
- **F1 score** - a single number balancing precision and recall together, useful when
  you want one summary metric rather than juggling two.
- **ROC-AUC** - measures how well the model's predicted *probabilities* rank churners
  above non-churners across every possible decision threshold, not just the default
  0.5 cutoff. A score of 0.5 means no better than random guessing; 1.0 means perfect
  separation.

There's usually a real trade-off between precision and recall - a model that predicts
"churn" more aggressively will catch more true churners (higher recall) but also
raise more false alarms (lower precision). Which one to prioritize is a business
decision, not just a technical one - for churn prediction, missing an actual churner
(low recall) is often more costly than a false alarm, since a false alarm just means
offering a retention discount to someone who wasn't going to leave anyway.

In [13]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f'Precision: {precision:.4f}')
print(f'Recall:    {recall:.4f}')
print(f'F1 score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')

Precision: 0.6584
Recall:    0.5668
F1 score:  0.6092
ROC-AUC:   0.8418


An ROC-AUC comfortably above 0.5 tells you the model has genuinely learned something
useful - it's meaningfully better than random guessing at ranking who's likely to
churn. Whether the precision/recall balance is *good enough* depends entirely on the
business context, which is exactly the kind of judgment call Domain 2 wants you to be
able to reason through, not just compute.

---

**That's it for today.** Small, complete increment:
- loaded the processed dataset from notebook 04
- split into train/test sets with `stratify=y` to preserve class balance
- scaled features correctly (fit on train only, applied to test)
- trained a first Logistic Regression model
- evaluated it properly: confusion matrix, precision, recall, F1, and ROC-AUC - and
  understood *why* accuracy alone would have been misleading

**Next session:** we'll try a second, more powerful algorithm (likely Random Forest)
to compare against this baseline, and look at *feature importance* - which specific
features the model is actually relying on to make its predictions.